In [ ]:
# %%
# =========================
# 1. Imports & device
# =========================
import json
import os
import re
from tqdm import tqdm

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    get_cosine_schedule_with_warmup
)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# GPU / CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
# %%
# =========================
# 2. Chargement JSONL
# =========================
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

train_raw = load_jsonl("../../data/train.jsonl")
test_raw  = load_jsonl("../../data/kaggle_test.jsonl")

print("Taille train_raw:", len(train_raw))
print("Taille test_raw :", len(test_raw))


In [ ]:
# %%
# =========================
# 3. Extraction du texte
# =========================
def extract_text(tweet):
    """
    Priorité :
    1) extended_tweet.full_text
    2) text
    3) quoted_status.extended_tweet.full_text
    4) quoted_status.text
    """
    if "extended_tweet" in tweet and tweet["extended_tweet"]:
        if "full_text" in tweet["extended_tweet"]:
            return tweet["extended_tweet"]["full_text"]

    if "text" in tweet and tweet["text"]:
        return tweet["text"]

    if "quoted_status" in tweet and tweet["quoted_status"]:
        q = tweet["quoted_status"]
        if "extended_tweet" in q and q["extended_tweet"] and "full_text" in q["extended_tweet"]:
            return q["extended_tweet"]["full_text"]
        if "text" in q:
            return q["text"]

    return ""


In [ ]:
# %%
# =========================
# 4. Compteurs de base
# =========================
def count_hashtags(entities):
    try:
        return len(entities.get("hashtags", []))
    except:
        return 0

def count_urls(entities):
    try:
        return len(entities.get("urls", []))
    except:
        return 0

def count_user_mentions(entities):
    try:
        return len(entities.get("user_mentions", []))
    except:
        return 0

def safe_get(d, key, default=0):
    return d.get(key, default) if isinstance(d, dict) else default


In [ ]:
# %%
# =========================
# 5. Features "user" & "quoted_status.user"
# =========================
def extract_user_features(user):
    """
    Retourne un dict de features numériques pertinents pour le user.
    """
    if user is None or not isinstance(user, dict):
        return {
            "user_followers": 0,
            "user_friends": 0,
            "user_statuses": 0,
            "user_verified": 0,
            "user_listed": 0,
            "user_favourites": 0,
            "user_geo_enabled": 0,
        }

    return {
        "user_followers": safe_get(user, "followers_count", 0),
        "user_friends": safe_get(user, "friends_count", 0),
        "user_statuses": safe_get(user, "statuses_count", 0),
        "user_verified": int(safe_get(user, "verified", False)),
        "user_listed": safe_get(user, "listed_count", 0),
        "user_favourites": safe_get(user, "favourites_count", 0),
        "user_geo_enabled": int(safe_get(user, "geo_enabled", False)),
    }


def extract_quoted_user_features(quoted_status):
    """
    Features du user de quoted_status (si présent).
    """
    if quoted_status is None or "user" not in quoted_status:
        return {
            "q_user_followers": 0,
            "q_user_friends": 0,
            "q_user_statuses": 0,
            "q_user_verified": 0,
        }
    u = quoted_status["user"]

    return {
        "q_user_followers": safe_get(u, "followers_count", 0),
        "q_user_friends": safe_get(u, "friends_count", 0),
        "q_user_statuses": safe_get(u, "statuses_count", 0),
        "q_user_verified": int(safe_get(u, "verified", False)),
    }


In [ ]:
# %%
# =========================
# 6. Features du tweet
# =========================
def extract_tweet_features(t):
    entities = t.get("entities", {})
    quoted  = t.get("quoted_status", None)

    feats = {
        "len_text": len(extract_text(t)),
        "has_extended": int("extended_tweet" in t and t["extended_tweet"] is not None),
        "n_hashtags": count_hashtags(entities),
        "n_urls": count_urls(entities),
        "n_mentions": count_user_mentions(entities),

        "favorite_count": safe_get(t, "favorite_count", 0),
        "retweet_count": safe_get(t, "retweet_count", 0),
        "reply_count": safe_get(t, "reply_count", 0),
        "quote_count": safe_get(t, "quote_count", 0),
        # "possibly_sensitive": int(safe_get(t, "possibly_sensitive", False)),   # supprimé
        "is_quote_status": int(safe_get(t, "is_quote_status", False)),
        # "lang_fr": int(safe_get(t, "lang", "") == "fr"),                       # supprimé
    }

    # User features
    feats.update(extract_user_features(t.get("user", None)))

    # Quoted status features
    feats.update({
        "quoted_exists": int(quoted is not None),
    })

    # Quoted user features
    feats.update(extract_quoted_user_features(quoted))

    return feats


In [ ]:
# %%
# =========================
# 7. Construction DataFrames
# =========================
def build_dataframe(raw, is_train=True):
    rows = []
    for tw in raw:
        text = extract_text(tw)
        feats = extract_tweet_features(tw)
        feats["text"] = text
        feats["challenge_id"] = tw["challenge_id"]
        if is_train:
            feats["label"] = tw["label"]
        rows.append(feats)
    return pd.DataFrame(rows)

df_train = build_dataframe(train_raw, is_train=True)
df_test  = build_dataframe(test_raw,  is_train=False)

print(df_train.head())
print("Colonnes train :", len(df_train.columns))
print("Colonnes test  :", len(df_test.columns))


In [ ]:
# %%
# =========================
# 8. Num features + split + scaling
# =========================
# Toutes les colonnes sauf texte, label, id
num_features = [
    c for c in df_train.columns
    if c not in ["text", "label", "challenge_id"]
]

# On retire explicitement possibly_sensitive et lang_fr si jamais présents
REMOVE = ["possibly_sensitive", "lang_fr"]
num_features = [c for c in num_features if c not in REMOVE]

print("Features numériques utilisées (", len(num_features), "):")
print(num_features)

# Split train / val
df_tr, df_val = train_test_split(
    df_train,
    test_size=0.1,
    stratify=df_train["label"],
    random_state=42
)

# StandardScaler : fit sur TRAIN uniquement
scaler = StandardScaler()
scaler.fit(df_tr[num_features])

df_tr[num_features]  = scaler.transform(df_tr[num_features])
df_val[num_features] = scaler.transform(df_val[num_features])
df_test[num_features] = scaler.transform(df_test[num_features])

# Quelques checks
assert not df_tr[num_features].isna().any().any(),  "NaNs in TRAIN"
assert not df_val[num_features].isna().any().any(), "NaNs in VAL"
assert not df_test[num_features].isna().any().any(), "NaNs in TEST"

df_tr.head()


In [ ]:
# %%
# =========================
# 9. Tokenizer & Dataset
# =========================
MODEL_NAME = "vinai/bertweet-base"

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False,       # BERTweet n’a pas de fast tokenizer officiel
    normalization=True    # important pour les tweets (emoji, accents)
)

class TweetDataset(Dataset):
    def __init__(self, df):
        self.texts = df["text"].tolist()
        self.features = df[num_features].values.astype(np.float32)
        self.labels = df["label"].tolist() if "label" in df else None

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]

        enc = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt"
        )

        x = {k: v.squeeze(0) for k, v in enc.items()}
        feats = self.features[idx]

        if self.labels is None:
            return x, feats
        else:
            return x, feats, self.labels[idx]


In [ ]:
# %%
# =========================
# 10. Modèle Fusion BERTweet + features
# =========================
from transformers import RobertaModel

class FusionModel(nn.Module):
    def __init__(self, model_name, n_features, dropout=0.2):
        super().__init__()

        # BERTweet est basé sur RoBERTa
        self.encoder = RobertaModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size

        self.mlp = nn.Sequential(
            nn.Linear(hidden + n_features, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 1)   # sortie binaire (logit)
        )

    def forward(self, x, feats):
        out = self.encoder(
            input_ids=x["input_ids"],
            attention_mask=x["attention_mask"]
        )
        cls = out.last_hidden_state[:, 0]  # token [CLS]
        z = torch.cat([cls, feats], dim=1)
        logits = self.mlp(z)
        return logits


In [ ]:
# %%
# =========================
# 11. DataLoaders
# =========================
train_ds = TweetDataset(df_tr)
val_ds   = TweetDataset(df_val)

train_dl = DataLoader(train_ds, batch_size=16, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=32, shuffle=False)

len(train_dl), len(val_dl)


In [ ]:
# %%
# =========================
# 12. Setup entraînement
# =========================
model = FusionModel(MODEL_NAME, len(num_features)).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5,          # un peu plus haut que DeBERTa
    weight_decay=0.01
)

num_epochs = 3
total_steps = num_epochs * len(train_dl)
warmup_steps = int(0.05 * total_steps)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print("Total steps :", total_steps)
print("Warmup steps:", warmup_steps)


In [ ]:
# %%
# =========================
# 13. Boucle d'entraînement
# =========================
def run_epoch(dl, train=True, log_every=500):
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    correct = 0
    total = 0
    step = 0

    for batch in tqdm(dl):
        step += 1

        if train:
            optimizer.zero_grad()

        if train:
            x, feats, y = batch
        else:
            x, feats, y = batch

        x = {k: v.to(device) for k, v in x.items()}
        feats = feats.to(device)
        y = torch.tensor(y).float().unsqueeze(1).to(device)

        with torch.set_grad_enabled(train):
            logits = model(x, feats)
            loss = criterion(logits, y)

            if train:
                loss.backward()
                optimizer.step()
                scheduler.step()

        batch_size = len(y)
        total_loss += loss.item() * batch_size
        preds = (logits.sigmoid() > 0.5).long()
        correct += (preds == y.long()).sum().item()
        total += batch_size

        if step % log_every == 0:
            current_loss = total_loss / total
            current_acc  = correct / total
            split = "TRAIN" if train else "VAL"
            print(f"[{split} step {step}] loss={current_loss:.4f} acc={current_acc:.4f}")

    epoch_loss = total_loss / total
    epoch_acc  = correct / total
    return epoch_loss, epoch_acc


for epoch in range(num_epochs):
    print(f"\n===== EPOCH {epoch+1}/{num_epochs} =====")
    train_loss, train_acc = run_epoch(train_dl, train=True, log_every=500)
    val_loss, val_acc     = run_epoch(val_dl,   train=False, log_every=500)

    print(f"Train: loss={train_loss:.4f} acc={train_acc:.4f}")
    print(f"Val  : loss={val_loss:.4f} acc={val_acc:.4f}")


In [ ]:
# %%
# =========================
# 14. Prédiction sur le test set
# =========================
test_ds = TweetDataset(df_test)
test_dl = DataLoader(test_ds, batch_size=32, shuffle=False)

all_preds = []
all_ids = df_test["challenge_id"].tolist()

model.eval()
with torch.no_grad():
    for batch in tqdm(test_dl):
        x, feats = batch
        x = {k: v.to(device) for k, v in x.items()}
        feats = feats.to(device)

        logits = model(x, feats)
        probs = logits.sigmoid().cpu().numpy().flatten()
        all_preds.extend(probs.tolist())

len(all_preds), len(all_ids)


In [ ]:
# %%
# =========================
# 15. Agrégation & Submission
# =========================
df_pred = pd.DataFrame({
    "challenge_id": all_ids,
    "prob": all_preds,
})

# Agrégation au niveau user
agg = df_pred.groupby("challenge_id")["prob"].mean().reset_index()
agg["Prediction"] = (agg["prob"] > 0.5).astype(int)

submission = agg[["challenge_id", "Prediction"]].copy()
submission.columns = ["ID", "Prediction"]

submission.head()
# %%
# Sauvegarde CSV pour Kaggle
submission.to_csv("submission_bertweet_fusion.csv", index=False)
print("✅ Fichier 'submission_bertweet_fusion.csv' généré !")
